# Initalize libraries

## Import libraries

In [ ]:
import sys, os
import time
from os.path import join
from os import path
from importlib import reload
from getpass import getuser
from tqdm.auto import tqdm

#data
import numpy as np
import xarray as xr
import h5py
from PIL import Image

# plotting
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from matplotlib.colors import LogNorm

# pyFAI
import pyFAI
from pyFAI.azimuthalIntegrator import AzimuthalIntegrator
from pyFAI.detectors import Detector

# Self-written libraries
sys.path.append(join(os.getcwd(), "library"))
import helper_functions as helper
import interactive
from interactive import cimshow
import mask_lib

# Gifs
import imageio

plt.rcParams["figure.constrained_layout.use"] = True  # replaces plt.tight_layout

In [ ]:
# Is there a GPU?
try:
    # Cupy
    import cupy as cp
    import cupyx as cpx

    GPU = True

    print("GPU available")

    # Self-written library
    import CCI_core_cupy as cci
except:
    GPU = False
    import CCI_core as cci

    print("GPU unavailable")

In [ ]:
# interactive plotting
import ipywidgets

%matplotlib widget

# Auto formatting of cells
#%load_ext jupyter_black

## Experiment specific Functions

In [ ]:
facility = "EuXFEL" # Options: "SwissFEL", "MAXI"
PROPOSALID = 10582
BEAMTIMEID = 202601 # Proposal number
data_fname_prefix = "2604_softimax"
USER = getuser()

# Facility specific loading functions
if facility == "PETRA":
    import PETRA_MaxP04_loading as loading
elif facility == "MAXI":
    import MAXI_loading as loading
elif facility == "SwissFEL":
    from sfdata import SFDataFiles, SFScanInfo, SFProcFile
    import Swiss_FEL_Loading as loading

    # Number or jobs for analysis
    NR_JOBS = 32
elif facility == "MAXIV":
    import MAXI_loading as loading
elif facility == "EuXFEL":
    import EuXFEL_loading as loading
    import toolbox_scs as tb
    
BASEFOLDER = f"/gpfs/exfel/d/raw/SCS/{BEAMTIMEID:d}/p{PROPOSALID:06d}"
DATAFOLDER = join(BASEFOLDER)

print("Raw Datafolder is: %s"%DATAFOLDER)

# Load dictionary for keys etc
mnemonics = tb.mnemonics
mnemonics_we = loading.load_mnemonics()
#mnemonics.update(loading.beamtime_mnemonics())

### Loading data

In [ ]:
def load_key(scan_id, key):
    """
    Load any kind of data specified by key (path)
    
    Parameter
    =========
    scan_id : int
        experimental identifier of scan
    key : str
        key path of nexus file tree to relevant data field
   
    Output
    ======
    data : dict
        data dictionaray on single key
    ======
    author: ck 2024, sw 2026
    """
    
    # load data with basic loading function
    data = loading.load_key(PROPOSALID, scan_id, key)
    
    return data

def load_data(scan_id, keys):
    """
    Load data of all specified keys from keypath

    Parameter
    =========
    scan_id : int
        experimental identifier of scan
    keypath : str
        path of nexus file tree to relevant data field
    keys : str or list of strings
        keys to load from keypath

    Output
    ======
    data : dict
        data dictionary of keys
    ======
    author: ck 2024, sw 2026
    """

    # load data with basic loading function
    run, data = tb.load(PROPOSALID,scan_id,keys)

    return data

## Loading images

In [ ]:
import numpy as np


def binning_func(images: np.ndarray, binning: int = 2) -> np.ndarray:
    """
    Rearrange image data by splitting the width into `binning` chunks and
    interleaving those chunks along the height axis.

    Parameters
    ----------
    images : np.ndarray
        Input array with shape (n_images, height, width).
    binning : int
        Binning factor.

    Returns
    -------
    np.ndarray
        Output array with shape:
        (n_images, height // binning, width // binning)
    """
    if images.ndim != 3:
        raise ValueError("images must be a 3D array with shape (n_images, height, width).")

    if binning <= 0:
        raise ValueError("binning must be a positive integer.")

    n_images, height, width = images.shape

    output_height = height // binning
    output_width = width // binning
    source_height = output_height // binning

    source = images[:, :source_height, :]

    output = np.zeros(
        (n_images, output_height, output_width),
        dtype=images.dtype,
    )

    for offset in range(binning):
        start = offset * output_width
        end = start + output_width
        output[:, offset::binning, :] = source[:, :, start:end]

    return output

### Loading image procedure

In [ ]:
def load_run_data(run_id, fields):
    """
    Wraps tb.load without proposal id

    Parameter
    =========
    run_id : int
        run number of scan
    fields : str or list of strings
        fields to load from log file

    Output
    ======
    run : obj
        extra_data data class (EuXFEL)
    data : xarray
        loaded data
    ======
    """

    # load data with basic loading function
    run, data = tb.load(PROPOSALID, run_id, fields)

    return run , data

def load_images(run_id: int, img_indices = None):
    """
    Load images corresponding to a given experimental image ID.

    Parameters
    ----------
    run_id : int
        run number of scan

    Returns
    -------
    images : np.ndarray
        Image stack with shape (n_frames, height, width).

    Raises
    ------
    ValueError
        If there are no camera data available
    ======
    """

    run, data = tb.load(PROPOSALID,run_id,mnemonics_we["images"])
    images = data[mnemonics_we["images"]].values

    if img_indices is not None:
        images = images[np.atleast_1d(img_indices)]
    
    return images.astype("float")


In [ ]:
# Full image loading procedure
def load_processing(im_id, img_indices = None, binning = 1, crop = None):
    """
    Loads images, averaging of two individual images (scans in tango consist of two images),
    padding to square shape, Additional cropping (optional)
    """

    # Load data
    images = load_images(im_id, img_indices = img_indices)

    # Force into square shape
    #images = helper.make_square_shape(images)

    # Optional cropping
    if crop is not None:
        images = images[..., :crop, :crop]

    # Binning
    if binning > 1:
        images = helper.binning(images, binning)

    # Average over all images
    if images.ndim == 4:
        image = np.mean(images, axis=(0, 1))
    elif images.ndim == 3:
        image = np.mean(images, axis=(0))
    elif images.ndim == 2:
        image = images.copy()
    images = np.stack(images)
    
    return image, images

# Full image loading procedure
def image_processing(images, binning = 1, crop = None):
    """
    Loads images, averaging of two individual images (scans in tango consist of two images),
    padding to square shape, Additional cropping (optional)
    """

    # Force into square shape
    images = helper.make_square_shape(images)

    # Optional cropping
    if crop is not None:
        images = images[..., :crop, :crop]

    # Binning
    if binning > 1:
        images = helper.binning(images, binning)

    # Average over all images
    if images.ndim == 4:
        image = np.mean(images, axis=(0, 1))
    elif images.ndim == 3:
        image = np.mean(images, axis=(0))
    elif images.ndim == 2:
        image = images.copy()
    images = np.stack(images)
    
    return image, images

## Other

In [ ]:
def combine_scans(array):
    try: 
        array = np.stack(array)
    except:
        array = np.concatenate(array)
    return np.squeeze(array)

In [ ]:
def save_gif(output_path, image_path_list, fps=3 ):
    writer = imageio.get_writer(output_path, format="GIF-PIL", fps=fps)
    for im in tqdm(image_path_list):
        writer.append_data(imageio.imread(im))
    writer.close()

# Experimental Details

In [ ]:
# Get detector pixel size, CMOS: 11 um, Sophia CCD: 13.5 um, Other CCD: 20µm, P-MTE3: 15µm
experimental_setup = {
    "ccd_dist": 0.4,  # ccd to sample distance
    "px_size": 15e-6,  # 
    "binning": 1,  # Camera binning
    "oversaturation": 2**16,  # Pixel saturation threshold
}

# Setup for azimuthal integrator
detector = Detector(
    experimental_setup["binning"] * experimental_setup["px_size"],
    experimental_setup["binning"] * experimental_setup["px_size"],
)

# General saving folder and log folder
folder_general = f"/gpfs/exfel/u/usr/SCS/{BEAMTIMEID:d}/p{PROPOSALID:06d}/Analysis"
helper.create_folder(folder_general)

print("Output Folder: %s" % folder_general)

# Load images


In [ ]:
# Specify image ids in iterable like list or array
im_ids = [89]
topo_ids = None #[3298]*np.ones(len(im_ids),dtype=int)#*np.arange(3280,3295+1)

dark_ids_im = [87] #1404*np.ones(len(im_ids),dtype=int) #iterable; one needs to be assigned for each image
dark_ids_topo = dark_ids_im#3296*np.ones(len(im_ids),dtype=int)

# Which data from nexus files to load? (e.g. "energy","srotz", ...)
key = "energy"

# Sort data according to scan axis?
sort = False

In [ ]:
# Load data
data = []
for i, im_id in enumerate(tqdm(im_ids,desc="ImageId")):
    # load images
    tdata = load_data(im_id,[mnemonics_we["images"],mnemonics_we[key],mnemonics_we["energy"]])
    tdata = tdata.rename({mnemonics_we["images"]: "images",mnemonics_we[key]:key})

    if mnemonics_we["energy"] in tdata:
        tdata = tdata.rename({mnemonics_we["energy"]: "energy"})
    
    images = tdata["images"].values
    _, images = image_processing(images)
    tdata["images"] = tdata["images"].copy(data=images) 

    ## Load and subtract dark images
    if dark_ids_im is not None:
        dark_im, _ = load_processing(dark_ids_im[i], img_indices = None, crop=None)
        tdata["dark_im"] = xr.DataArray(dark_im,dims=["x","y"])
        tdata["images"] = tdata["images"] - tdata["dark_im"]

    # Load corresponding topo image:
    if topo_ids is not None:
        ttopo, _ = load_processing(topo_ids[i])
        tdata["topo"] = xr.DataArray(ttopo,dims=["x","y"])

        if dark_ids_topo is not None:
            dark_topo, _ = load_processing(dark_ids_topo[i], img_indices = None, crop=None)
            tdata["dark_topo"] = xr.DataArray(dark_topo,dims=["x","y"])
            tdata["topo"] = tdata["topo"] - tdata["dark_topo"]
    else:
        tdata["topo"] = xr.DataArray(np.zeros((len(tdata["x"]),len(tdata["y"]))),dims=["x","y"])

    tdata["wavelength"] = helper.photon_energy_wavelength(tdata["energy"])
    
    data.append(tdata)

# Combine into single xarray
print("Concatenating...")
data = xr.concat(data, "trainId", data_vars="different")
print("Done!")

# Plot scan axis
fig, ax = plt.subplots()
ax.plot(data[key], "o-")
ax.set_title("Scan Axis")
ax.set_ylabel(key)
ax.set_xlabel("Frame Index")
ax.grid()

In [ ]:
# Assign to xarray
data["image"] = data["images"].mean("trainId")
image = data["image"].values

# Sort ascending key
if sort is True:
    data = data.sortby(key)

# Assign im_id as attribute
data = data.assign_attrs({"im_ids": im_ids})

# Slideshow viewer
fig, ax = cimshow(data["images"], title=data[key].values)

In [ ]:
fig, ax = plt.subplots()
ax.plot(data[key],data["images"].mean(["y","x"]))

In [ ]:
fig, ax = cimshow(helper.log_clip(data["images"]),title=data[key].values)

In [ ]:
# Select roi for charge reference to determine diff images
x1, x2 = ax.get_xlim()
y2, y1 = ax.get_ylim()
roi = np.array([y1, y2, x1, x2]).astype(int)  # ystart, ystop, xstart, xstop
roi_s = np.s_[roi[0] : roi[1], roi[2] : roi[3]]

In [ ]:
# Plotting limits
#roi= np.s_[375:875,400:850]
#roi = np.s_[0:image.shape[0],0:images.shape[1]]

# Calculation of difference
use_diff = False
diff = np.zeros_like(data["images"].values)
folder_gif = helper.create_folder(join(folder_general, "%d-%d" % (im_ids[0],im_ids[-1])))
frame_paths = []
for i, im in enumerate(data["images"].values):
    if use_diff:
        # will be subtracted
        ref_image = data["topos"].values[i]
        #ref_image = data["images"][0]#.values[i]
    
        # Calc intensity scaling factor
        factor, offset = cci.dyn_factor(
        im[roi_s],
        ref_image[roi_s],
        method="correlation",
        verbose=True,
        plot=False)
    
        # Subtract topography
        diff[i] = im/factor - ref_image
    else:
        diff[i] = im
    
    # correct offset
    offset = np.mean(diff[:50])
    diff[i] = diff[i] - offset

# Plotting of diff
vmin, vmax = np.percentile(diff[:,*roi_s],(1,99))
vmax = np.max(np.abs([vmin,vmax]))
vmin = -vmax
binning = 1
for i, im in enumerate(tqdm(data["images"].values,desc="Saving")):    
    # Plot image
    plt.ioff()
    fig, ax = plt.subplots(figsize=(9,8))
    if topo_ids is not None:
        ax.set_title(f"ImId: {im_id:03d} - {topo_id:03d} @ {key}: {data[key].values[i]:.2f}")
    else:
        ax.set_title(f"ImId: {im_id:03d} @ {key}: {data[key].values[i]:.2f}")
    vmin, vmax = np.percentile(diff[i,*roi_s],(1,99))
    m = ax.imshow(helper.binning(diff[i,*roi_s],binning), vmin=vmin,vmax=vmax,cmap="viridis")
    #ax.invert_yaxis()
    plt.colorbar(m)
    
    # Save image
    fname = "SAXS_ImId_%04d_%04d_%03d_%s.png" % (im_ids[0], im_ids[-1], i, USER)

    # Save
    fname = path.join(folder_gif, fname)
    frame_paths.append(fname)
    plt.savefig(fname)

# Add also to xarray
data["diffs"] = xr.DataArray(diff,dims=["trainId","y","x"])
#sel = data.y < 25
#data["offset_diff"] = data["diffs"].where(sel).mean(["y","x"])
#data["diffs"] = data["diffs"] - data["offset_diff"]

# Create gif
output_gif = f"SAXS_{im_ids[0]}-{im_ids[-1]}.gif"
save_gif(join(folder_general,output_gif), frame_paths, fps=5)
plt.ion()
print("Done!")

In [ ]:
interactive.cimshow(data["diffs"][:,*roi_s],title=data[key].values,cmap="viridis")

In [ ]:
cimshow(data["diffs"][:,*roi],cmap="coolwarm")

# Draw beamstop mask

In [ ]:
poly_mask = interactive.draw_polygon_mask(image)

In [ ]:
# Take poly coordinates and mask from widget
p_coord = poly_mask.get_vertice_coordinates()
mask_draw = poly_mask.full_mask.astype(int)

print("Copy these coordinates into the 'load_poly_coordinates()' function:")
print(p_coord)

# Plot image with beamstop and valid pixel mask
fig, ax = plt.subplots(1, 3, sharex=True, sharey=True, figsize=(9, 3))
tmp = image * (1 - mask_draw)
mi, ma = np.percentile(tmp[tmp != 0], [0.1, 99.9])
ax[0].imshow(image * (1 - mask_draw), cmap="viridis", vmin=mi, vmax=ma)
ax[0].set_title("Image * (1-mask_draw)")

mi, ma = np.percentile(image * mask_draw, [0.1, 99.9])
ax[1].imshow(image * mask_draw, vmin=mi, vmax=ma)
ax[1].set_title("Image * mask_draw")

ax[2].imshow(1 - mask_draw)
ax[2].set_title("1 - mask_draw")

In [ ]:
def load_poly_coordinates():
    """
    Dictionary that stores polygon corner coordinates of all drawn masks
    Example: How to add masks with name "test":
    mask_coordinates["test"] = copy coordinates from above
    """
    mask_coordinates = dict()
    mask_coordinates["membranes"] = [[(800.2, 995.6), (751.9, 999.7), (747.8, 1043.3), (799.0, 1049.1)], [(885.6, 998.6), (881.5, 1052.8), (945.0, 1059.3), (949.2, 1001.0)], [(1026.0, 1139.4), (1022.5, 1200.7), (1073.7, 1204.2), (1073.7, 1137.1)], [(1018.9, 1283.1), (1017.8, 1334.3), (1069.0, 1337.8), (1070.1, 1281.9)], [(1061.5, 1425.6), (1016.8, 1426.7), (1016.5, 1468.9), (1059.8, 1469.1)], [(1223.3, 1015.5), (1165.5, 1011.7), (1163.6, 1057.9), (1216.2, 1061.7)], [(1361.8, 1012.3), (1305.3, 1016.8), (1305.3, 1066.8), (1356.0, 1068.1)], [(1443.8, 1012.1), (1443.8, 1065.7), (1493.6, 1067.7), (1493.6, 1017.3)], [(1630.1, 1030.0), (1579.7, 1033.8), (1582.2, 1069.6), (1629.5, 1072.1)], [(1768.9, 1032.5), (1723.0, 1031.9), (1721.5, 1075.9), (1766.8, 1076.3)]]
    return mask_coordinates

In [ ]:
# Which drawn masks do you want to load? Use can combine multiple masks, e.g., ["bs_left_part", "bs_bot_part", "bs_top_part"]
polygon_names = ["membranes"]
mask_draw = mask_lib.load_poly_masks(
    experimental_setup["binning"] * image.shape,
    load_poly_coordinates(),
    polygon_names,
)

# optional binning
mask_draw = helper.binning(mask_draw,experimental_setup["binning"])

# Use widget to shift and expand or shrink the mask
ss_mask = interactive.Shift_Scale_Mask(image, mask_draw, shift=[0,1], scale=-1)

In [ ]:
# Take mask, shift and scaling from widget
mask, mask_shift, mask_scale = ss_mask.get_mask()

# Add to xarray
data["mask"] = xr.DataArray(mask, dims=["y", "x"])

# Plot image with beamstop and valid pixel mask
fig, ax = plt.subplots(1, 3, sharex=True, sharey=True, figsize=(9, 3))
mi, ma = np.percentile(image, [0.1, 99.9])
ax[0].imshow(image, cmap="viridis", vmin=mi, vmax=ma)
ax[0].set_title("Image * (1-mask)")

mi, ma = np.percentile(image * mask, [0.1, 99.9])
ax[1].imshow(image * mask, vmin=mi, vmax=ma)
ax[1].set_title("Image * mask")

ax[2].imshow(1 - mask)
ax[2].set_title("1 - mask")
plt.tight_layout()

# Find center

## Basic widget to find center

Try to **align** the circles to the **center of the scattering pattern**. Care! Position of beamstop might be misleading and not represent the actual center of the hologram. 

In [ ]:
# Set center position via widget
c0, c1 = [1040, 1051] # initial values
ic = interactive.InteractiveCenter(data["image"], c0=c0, c1=c1)

In [ ]:
# Get center positions
center = [ic.c0, ic.c1]
print(f"Center:", center)

## Azimuthal integrator widget for finetuning
If scattering pattern is radial symmetric, move center position until scattering ring is a line after transformation in polar coordinates

In [ ]:
# Setup azimuthal integrator for virtual geometry
ai = AzimuthalIntegrator(
    dist=experimental_setup["ccd_dist"],
    detector=detector,
    wavelength=data.wavelength.values[0],
    poni1=center[0]
    * experimental_setup["px_size"]
    * experimental_setup["binning"],  # y (vertical)
    poni2=center[1]
    * experimental_setup["px_size"]
    * experimental_setup["binning"],  # x (horizontal)
)

In [ ]:
# Calc azimuthal integration
I_t, q_t, phi_t = ai.integrate2d(
    image,
    700,
    #radial_range=(0.0, .3),
    unit="q_nm^-1",
    correctSolidAngle=True,
    dummy=np.nan,
    mask=mask,
    #method = "BBox"
)
az2d = xr.DataArray(I_t, dims=("phi", "q"), coords={"q": q_t, "phi": phi_t})

# Plot
fig, ax = plt.subplots()
mi, ma = np.nanpercentile(I_t, [1, 98])
az2d.plot.imshow(ax=ax, vmin=mi, vmax=ma)
plt.title(f"Azimuthal integration")

# Azimuthal integration for all images

In [ ]:
# Update center of azimuthal integrator
ai = AzimuthalIntegrator(
    dist=experimental_setup["ccd_dist"],
    detector=detector,
    wavelength=data.wavelength.values[0],
    poni1=center[0]
    * experimental_setup["px_size"]
    * experimental_setup["binning"],  # y (vertical)
    poni2=center[1]
    * experimental_setup["px_size"]
    * experimental_setup["binning"],  # x (horizontal)
)

In [ ]:
# Do 2d Azimuthal integration of all images and append them to list
list_q, list_i2d, list_i2d_diff = [], [], []

timages = data["images"].values
tdiffs = data["images"].values

for i, _ in enumerate(tqdm(timages,desc="SAXS images")):
    # Adapt azimuthal integrator if scan is an energy scan
    if key == "energy":
        ai.wavelength = helper.photon_energy_wavelength(data["energy"][i].values)

    # Calc ai for normal images
    i2d, q, chi = ai.integrate2d(
        timages[i],
        500,
        90,
        radial_range=(0.0, 0.1),
        unit="q_nm^-1",
        correctSolidAngle=True,
        dummy=np.nan,
        mask=mask,
        method = "BBox"
    )
    list_i2d.append(i2d)
    list_q.append(q)

    # Calc ai for diff images
    i2d, _, _ = ai.integrate2d(
        tdiffs[i],
        500,
        90,
        radial_range=(0.0, 0.1),
        unit="q_nm^-1",
        correctSolidAngle=True,
        dummy=np.nan,
        mask=mask,
        method = "BBox"
    )
    list_i2d_diff.append(i2d)

# Add to xarrays
data["q"] = q
data["chi"] = chi
data["i2d"] = xr.DataArray(list_i2d, dims=["trainId", "chi", "q"])
data["i2d_diff"] = xr.DataArray(list_i2d_diff, dims=["trainId", "chi", "q"])

## Select relevant chi-range

In [ ]:
cimshow(data["i2d_diff"],aspect="auto")

In [ ]:
# Plot 2d and 1d azimuthal integration to estimate the relevant chi and q range
# which image to show?
idx = 100

# Select chi-range
# Which chi-mode? ("all","other")
chi_mode = "other"

# Select chi-range
if chi_mode == "all":
    sel_chi = (data.chi <= 180) * (data.chi >= -180)
elif chi_mode == "other":
    sel_chi = (
        (data.chi <= 180) * (data.chi >= 155)
        + (data.chi <= 95) * (data.chi >= 65)
        + (data.chi <= 5) * (data.chi >= -20)
        + (data.chi <= -85) * (data.chi >= -110)
    )
data["i1d"] = (data.i2d.where(sel_chi, drop=True)).mean("chi")
data["i1d_diff"] = (data.i2d_diff.where(sel_chi, drop=True)).mean("chi")

# Plot
fig, ax = plt.subplots(
    2,
    1,
    figsize=(8, 8),
    sharex=True,
)
mi, ma = np.nanpercentile(data["i2d_diff"][idx], [1, 99])
data["i2d_diff"][idx].plot.imshow(ax=ax[0], vmin=mi, vmax=ma)
ax[0].set_title(f"2d Azimuthal integration")
ax[0].grid()

# Plot 1d azimuthal integration to estimate the relevant q-range
ax[1].plot(data.q, data.i1d_diff[idx])
#ax[1].set_yscale("log")
ax[1].set_title("1d Azimuthal Integration")
ax[1].grid()
ax[1].set_ylabel("Integrated intensity")
ax[1].set_xlabel("q")

## Select relevant q-range

In [ ]:
# Select relevant q-range for averaging
q0, q1 = 0.0, 0.1
#q0, q1 = 0, 0.1
binning = False
bins = []

# Get SAXS from q-range
sel = (data.q > q0) * (data.q < q1)

data["saxs"] = data.i1d.where(sel, drop=True).mean("q")
data["saxs_diff"] = data.i1d_diff.where(sel, drop=True).mean("q")

# Averaging of same scan axis values or binning
if binning is True:
    # Execute binning
    data_bin = data.groupby_bins(key, bins).mean()

    # Rename binned values, drop intervals as those cannot be save in h5
    bin_scan_axis = scan_axis + "_bins"
    data_bin = data_bin.swap_dims({bin_scan_axis: key})
    data_bin = data_bin.drop(bin_scan_axis)
else:
    _, count = np.unique(data[key].values, return_counts=True)
    if np.any(count > 1):
        data_bin = data.groupby(key).mean()
    else:
        data_bin = data.swap_dims({"trainId": key})
        

In [ ]:
# Plot Averaged 1d Intensity over q for all different scan 
fig, ax = plt.subplots()
colors = plt.cm.jet(np.linspace(0,1,len(data_bin[key])))

for i, i1dchi in enumerate(data_bin["i1d_diff"].values):
    ax.plot(data_bin["q"],i1dchi,label="%s: %.1f"%(key,data_bin[key][i]),color = colors[i])
    
ax.grid()
ax.legend(ncol = 2,fontsize = 8)
ax.set_xlabel("q")
ax.set_ylabel("Averaged Intensity")
#ax.set_yscale("log")
ax.set_xlim([0,q1])
ax.set_ylim([0, 1.1*data_bin["i1d_diff"].max().values])

## Title and fname
if len(im_ids) > 1:
    ax.set_title("Scan Id %s-%s" % (im_ids[0], im_ids[-1]))
    fname = "SAXS_i1d_ImId_%04d-%04d_%s.png" % (im_ids[0], im_ids[-1], USER)
else:
    ax.set_title("Scan Id %d" % (im_ids[0]))
    fname = "SAXS_i1d_ImId_%04d_%s.png" % (im_ids[0], USER)

fname = join(folder_general, fname)
print("Saving:%s" % fname)
plt.savefig(fname)

# Plotting

In [ ]:
# Plot Intensity of SAXS Pattern
fig, ax = plt.subplots()
ax.plot(data_bin[key].values, data_bin["saxs_diff"].values,"o-")
ax.grid()
ax.set_xlabel(key)
ax.set_ylabel("Integrated SAXS")

## Title and fname
if len(im_ids) > 1:
    ax.set_title("Scan Id %s-%s" % (im_ids[0], im_ids[-1]))
    fname = "SAXS_ImId_%04d-%04d_%s.png" % (im_ids[0], im_ids[-1], USER)
else:
    ax.set_title("Scan Id %d" % (im_ids[0]))
    fname = "SAXS_ImId_%04d_%s.png" % (im_ids[0], USER)

fname = join(folder_general, fname)
print("Saving:%s" % fname)
plt.savefig(fname)

# Export scan as gif

## Select roi of images for plotting

How to use:
1. Zoom into the image and adjust your FOV until you are satisfied.
2. Save the axes coordinates.

In [ ]:
fig, ax = cimshow(data_bin["diffs"].values)

In [ ]:
# Takes start and end of x and y axis
roi = interactive.axis_to_roi(ax)
print(f"Image registration roi:", roi)

## Plotting

In [ ]:
# Plot qrange
q0_plot, q1_plot = 0.0, 0.1

# Setup gif
folder_gif = helper.create_folder(join(folder_general, "ImId_%05d" % im_id))
variable_images_1d = []

# Find global max and min all images
allmin, allmax = np.nanpercentile(data_bin["i2d_diff"].values, [0.1, 100])
allImin = data_bin.i1d_diff.where(sel, drop=True).min()
allImax = data_bin.i1d_diff.where(sel, drop=True).max()

#if allImin <0.01:
#    allImin = 0.1
#if allmin <0.1:
#    allmin = 0.01

# Loop over images
for i in tqdm(range(len(data_bin[key].values))):
    # Plot for averaged image
    fig = plt.figure(figsize=(6, 10))
    gs1 = gridspec.GridSpec(
        4,
        1,
        figure=fig,
        left=0.2,
        bottom=0.05,
        right=0.975,
        top=1.1,
        wspace=0,
        hspace=0,
        height_ratios=[6, 1, 2, 1],
    )

    # Plot image roi
    ax0 = fig.add_subplot(gs1[0])
    tmp = data_bin["diffs"][i].values[roi]
    #tmp[tmp<allmin] = allmin
    #m = ax0.imshow(data_bin["diffs"][i].values[roi], norm = LogNorm(vmin = allImin, vmax=allImax))
    m = ax0.imshow(data_bin["diffs"][i].values[roi], vmin = allImin, vmax=allImax,cmap="coolwarm")
    plt.colorbar(m, ax=ax0, pad=0.045, location="bottom")

    # Plot 1d azimuthal integration
    ax1 = fig.add_subplot(gs1[1])
    tmp = data_bin.i1d_diff[i]
    ax1.plot(data_bin.q, tmp)
    ax1.set_xlabel("q")
    ax1.set_ylabel("Mean Intensity")
    ax1.set_xlim([q0_plot, q1_plot])
    ax1.set_ylim([allImin, 1.1*allImax])
    #ax1.set_yscale("log")
    ax1.grid()

    # Contor plot
    ax2 = fig.add_subplot(gs1[2])
    vmin, vmax = np.nanpercentile(data_bin["i1d_diff"], [1, 99.9])
    data_bin["i1d_diff"].plot.contourf(
        x=key,
        y="q",
        ax=ax2,
        cmap="viridis",
        add_colorbar=False,
        vmin=vmin,
        vmax=vmax,
        levels=200,
        ylim = [q0_plot,q1_plot]
    )
    ax2.vlines(data_bin[key].values[i], q0, q1,'r')
    ax2.hlines(q0, data_bin[key].min(),data_bin[key].max(),'w',linestyles='dashed')
    ax2.hlines(q1, data_bin[key].min(),data_bin[key].max(),'w',linestyles='dashed')

    # Plot SAXS Intensity
    ax3 = fig.add_subplot(gs1[3])
    ax3.plot(data_bin[key].values, data_bin["saxs_diff"].values)
    ax3.scatter(data_bin[key].values[i], data_bin["saxs_diff"].values[i], 20, color="r")
    ax3.set_xlabel(key)
    ax3.set_ylabel("Mean intensity")
    ax3.grid()
    ax3.set_xlim(data_bin[key].min(),data_bin[key].max())

    # Title and fname
    ax0.set_title(
        f"%04d - %04d %s = %s"
        % (im_ids[0], im_ids[-1], key, np.round(data_bin[key].values[i], 3))
    )
    fname = "SAXS_ImId_%04d_%04d_%03d_%s.png" % (im_ids[0], im_ids[-1], i, USER)

    # Save
    fname = path.join(folder_gif, fname)
    variable_images_1d.append(fname)
    plt.savefig(fname)
    plt.close()

# Create gif for 1d AI
if len(im_ids) > 1:
    fname = f"SAXS_ImId_%04d_%04d_%s.gif" % (im_ids[0], im_ids[-1], USER)
else:
    fname = f"SAXS_ImId_%04d_%s.gif" % (im_ids[0], USER)

gif_path = path.join(folder_general, fname)
print("Saving gif:%s" % gif_path)
helper.create_gif(variable_images_1d,gif_path,fps=4)
print("Done!")

In [ ]:
# Drop images
data_bin_save = data_bin.drop_vars(["images"])

# Save log
folder = join(folder_general, "Logs")
helper.create_folder(folder)
fname = join(folder, "SAXS_Log_ImId_%04d_%s.nc" % (im_id, USER))

print(f"Saving:", fname)
data_bin_save.to_netcdf(fname)